# Diffusion, for Images and for Text: Learn to Undo a Little Noise

The full code of [Diffusion, for Images and for Text: Learn to Undo a Little Noise](https://residual-stream.lvtan71.workers.dev/posts/diffusion-for-images-and-text/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/diffusion-for-images-and-text) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/diffusion-for-images-and-text.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 5 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q "numpy==2.*" "torch==2.14.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "diffusion-for-images-and-text"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `diff2d.py`

The linear noise schedule, then a noise-predicting network on the eight-mode ring of [VAEs and GANs](https://residual-stream.lvtan71.workers.dev/posts/vaes-and-gans/), trained on three seeds with a label dropped 10% of the time: DDPM and DDIM at 100 down to 2 steps, and classifier-free guidance at w = 0, 1, 3, 7. Saves the networks to `nets.pt`.

In [ ]:
!{sys.executable} diff2d.py

The post printed:

```text
linear schedule, beta from 1e-4 to 0.02 over T = 1000 steps
    t    abar_t  signal sqrt(abar)  noise sqrt(1-abar)       SNR
    1   0.99990             0.9999              0.0100      9999
   10   0.99811             0.9991              0.0435     526.8
  100   0.89702             0.9471              0.3209      8.71
  250   0.52409             0.7239              0.6899     1.101
  500   0.07859             0.2803              0.9599   0.08529
  750   0.00335             0.0579              0.9983  0.003362
 1000   0.00004             0.0064              1.0000 4.036e-05

unconditional samples (2,000 per run); modes covered, then share near a mode, per seed
DDPM, 1000 steps     8  8  8     84%   85%   85%
DDIM, 100 steps      8  8  8     83%   82%   81%
DDIM, 50 steps       8  8  8     81%   80%   80%
DDIM, 20 steps       8  8  8     76%   73%   73%
DDIM, 10 steps       8  8  8     58%   55%   51%
DDIM, 5 steps        0  0  1      3%    3%    3%
DDIM, 2 steps        0  1  0      2%    2%    2%

classifier-free guidance, DDIM 50 steps, 250 samples asked for each mode (2,000 per run)
   w   landed on the mode asked for   share near any mode
   0                10%   10%   11%       83%   81%   81%
   1               100%  100%  100%      100%  100%  100%
   3                89%   93%   89%       89%   93%   89%
   7                 4%    0%   12%        4%    0%   12%
```

## `ddim_spacing.py`

DDIM at 5, 10 and 20 steps with three choices of which timesteps to visit. Reads `diff2d.py`'s definitions and `nets.pt`.

In [ ]:
!{sys.executable} ddim_spacing.py

The post printed:

```text
unconditional, 2,000 samples per seed: share near a mode (modes covered)
  5 steps, even, ending at t = 0 (diff2d.py)    3% (0)    3% (0)    3% (1)   t = [999, 749, 500, 250, 0]
  5 steps, even, ending at t = 999/k           17% (8)   13% (7)   12% (8)   t = [999, 799, 599, 399, 199]
  5 steps, quadratic (DDIM paper)              50% (8)   42% (8)   39% (8)   t = [999, 639, 360, 160, 40]
 10 steps, even, ending at t = 0 (diff2d.py)   58% (8)   55% (8)   51% (8)
 10 steps, even, ending at t = 999/k           61% (8)   58% (8)   55% (8)
 10 steps, quadratic (DDIM paper)              67% (8)   65% (8)   63% (8)
 20 steps, even, ending at t = 0 (diff2d.py)   76% (8)   73% (8)   73% (8)
 20 steps, even, ending at t = 999/k           76% (8)   74% (8)   75% (8)
 20 steps, quadratic (DDIM paper)              77% (8)   74% (8)   74% (8)
```

## `guidance_radius.py`

Where guided samples go: median distance from the ring's center and spread around the mode asked for, at w = 1, 3, 7. Reads `diff2d.py`'s definitions and `nets.pt`.

In [ ]:
!{sys.executable} guidance_radius.py

The post printed:

```text
DDIM 50 steps, 250 samples per mode, all three seeds pooled
  w  median radius  spread around the mode asked for
  1           2.00                             0.032
  3           2.06                             0.081
  7           3.98                             4.737
```

## `masked_diffusion.py`

Masked diffusion on 90 eight-digit arithmetic sequences: a small bidirectional Transformer trained on three seeds, then decoded 1, 2, 4 or 8 tokens per step in three orders, against the exact first-step chance.

In [ ]:
!{sys.executable} masked_diffusion.py

The post printed:

```text
90 valid sequences of 8 digits; 3 models (seeds), 2,000 samples each

tokens per step  steps           order   valid samples, per seed     distinct valid (of 90)   first step, exact
              1      8   left to right    99.1%   98.5%   98.7%       90   90   90            100.00%
              1      8          random    99.1%   98.9%   98.3%       90   90   90            100.00%
              1      8  most confident    99.9%   99.8%   99.9%       90   90   90                   
              2      4   left to right    90.1%   89.5%   89.2%       90   90   90             90.00%
              2      4          random    58.1%   58.8%   58.2%       90   90   90             65.36%
              2      4  most confident    30.4%   88.4%   89.2%       88   90   90                   
              4      2   left to right     1.1%    0.9%    1.0%       22   19   18              0.90%
              4      2          random     1.1%    0.9%    0.5%       16   18    9              0.89%
              4      2  most confident     1.1%    1.0%    1.0%       19   16   19                   
              8      1     all at once     0.0%    0.0%    0.0%        0    0    0              0.00%
   1, 1, then 6      3   left to right    98.8%   98.0%   98.1%       90   90   90
   1, 1, then 6      3          random    55.1%   56.5%   54.8%       90   90   90
digits 1 apart: 9 of the 9 possible differences fix the step
digits 2 apart: 1 of the 5 possible differences fix the step
digits 3 apart: 9 of the 9 possible differences fix the step
digits 4 apart: 1 of the 5 possible differences fix the step
digits 5 apart: 0 of the 2 possible differences fix the step
digits 6 apart: 1 of the 5 possible differences fix the step
digits 7 apart: 9 of the 9 possible differences fix the step
```

## `flops.py`

Arithmetic: FLOPs and the latency floor of autoregressive decoding against masked diffusion for an 8B model on one H100.

In [ ]:
!{sys.executable} flops.py

The post printed:

```text
autoregressive: 512 steps,    8.2 TFLOP, at least  2.45 s
 diffusion steps S  tokens per step    TFLOP  x AR FLOPs  time at least
               512                1     4194         512         4.24 s
               256                2     2097         256         2.12 s
               128                4     1049         128         1.06 s
                64                8      524          64         0.53 s
                32               16      262          32         0.27 s
                16               32      131          16         0.13 s
```